# Causal effect estimation of a social navigation signalCompanion notebook to the BSc thesis *Causal Effect Estimation of Robot Actions forHuman Aware Navigation* (F. Baldo, University of Padua, 2026) and to the AIRO 2026paper *Estimating the Causal Impact of Social Navigation Signals in Corridor Scenarios*.It reproduces every number reported in Chapter 5 of the thesis from the raw episodedataset, and regenerates Figure 5.1.**Data.** `episodes_100_v1.csv` — 100 observational episodes, one row each, over the sixbinary DAG variables collected in the corridor scenario:| variable | meaning | 1 | 0 ||---|---|---|---|| `Pi` | pedestrian occupancy at decision time | congested | open || `A`  | robot action (LED signal) | signal | no signal || `Pe` | pedestrian occupancy after the response window | congested | open || `S`  | geometric clearance available for the transit | sufficient | insufficient || `T`  | task outcome | success | timeout || `O`  | confounder: static obstacles in the junction | present | absent |**Assumed DAG.** `Pi → A → Pe → S → T`, with the confounder entering at two points,`O → A` and `O → S`, which opens the single backdoor path `A ← O → S → T`.**Requirements.** `pandas`, `numpy`, `scipy`, `matplotlib` are required. `pyagrum` and`causal-learn` are *optional*: every estimate in the thesis is computed here in plainpandas, and the two libraries are used only for the independent cross-check(Section 6) and for the structure-learning step (Section 4). Cells that need them areguarded and skip cleanly if the import fails.```pip install pandas numpy scipy matplotlibpip install pyagrum causal-learn      # optional```

In [ ]:
import itertoolsimport numpy as npimport pandas as pdfrom scipy.stats import chi2, chi2_contingencyimport matplotlib.pyplot as pltpd.set_option("display.width", 100)# Optional dependencies. Everything essential runs without them.try:    import pyAgrum as gum    import pyAgrum.causal as csl    HAS_PYAGRUM = Trueexcept ImportError:    HAS_PYAGRUM = Falsetry:    from causallearn.search.ConstraintBased.PC import pc    from causallearn.utils.cit import chisq    HAS_CAUSALLEARN = Trueexcept ImportError:    HAS_CAUSALLEARN = Falseprint(f"pyAgrum      : {'available' if HAS_PYAGRUM else 'not installed (optional)'}")print(f"causal-learn : {'available' if HAS_CAUSALLEARN else 'not installed (optional)'}")# Every estimate produced below is collected here and printed as a table in Section 5.RESULTS = []

---## 1. The datasetThesis Section 5.1. The dataset is loaded as it was extracted from the ROS bags: one rowper episode, six binary variables, no missing values and no discarded episode. Theassertions below fail loudly if the file does not match the dataset the thesis reports on.

In [ ]:
NODES = ["Pi", "A", "Pe", "S", "T", "O"]data = pd.read_csv("episodes_100_v1.csv")data = data[NODES].astype(int)assert len(data) == 100, f"expected 100 episodes, found {len(data)}"assert not data.isna().any().any(), "missing values in the dataset"assert set(np.unique(data.values)) <= {0, 1}, "non-binary values in the dataset"print(f"Episodes: {len(data)}\n")print("Marginals P(.=1):")for c in NODES:    print(f"  P({c}=1) = {data[c].mean():.2f}")# Marginals reported in Section 5.1 of the thesis.EXPECTED = {"Pi": 0.48, "A": 0.38, "Pe": 0.25, "S": 0.73, "T": 0.76, "O": 0.29}for c, v in EXPECTED.items():    assert abs(data[c].mean() - v) < 5e-3, f"{c}: {data[c].mean():.3f} != {v}"print("\nAll marginals match the values reported in the thesis.")

The confounder marginal `P(O=1) = 0.29` is consistent with the Bernoulli parameter`P(O=0) = 0.7` used by the obstacle policy during collection.Two further checks reported in Section 5.1 — the delay between the `Pi` and `Pe` readings(0.0 s when `A=0`, 5.64 s when `A=1`) and the symmetry of the action and success ratesacross the two traversal directions — are computed from the per-timestep extraction, notfrom this aggregated file, and are therefore not reproduced here.

---## 2. Naive versus adjusted estimateThesis Section 5.2. The quantity of interest is the average treatment effect of theaction on the outcome,$$\mathrm{ATE} = P(T=1 \mid do(A=1)) - P(T=1 \mid do(A=0))$$The naive estimate reads the difference straight off the conditional success rates andadjusts for nothing.

In [ ]:
def success_table(df):    """Success rate stratified by O and A — Table 5.1 of the thesis."""    rows = []    for o in (0, 1):        for a in (0, 1):            g = df[(df.O == o) & (df.A == a)]            rows.append({"O": o, "A": a, "n": len(g),                         "successes": int(g["T"].sum()),                         "P(T=1)": g["T"].mean()})    for a in (0, 1):        g = df[df.A == a]        rows.append({"O": "all", "A": a, "n": len(g),                     "successes": int(g["T"].sum()),                     "P(T=1)": g["T"].mean()})    return pd.DataFrame(rows)tbl = success_table(data)print(tbl.to_string(index=False, formatters={"P(T=1)": "{:.3f}".format}))

In [ ]:
p_t_a1 = data[data.A == 1]["T"].mean()p_t_a0 = data[data.A == 0]["T"].mean()ate_naive = p_t_a1 - p_t_a0print(f"P(T=1 | A=1) = {p_t_a1:.3f}")print(f"P(T=1 | A=0) = {p_t_a0:.3f}")print(f"naive ATE    = {ate_naive:+.3f}")RESULTS.append(("---", "none", "naive, unadjusted", ate_naive))

Taken at face value this says the signal is harmful, lowering the success rate by roughly21 points. The association is spurious and entirely driven by the confounder.### 2.1 Backdoor adjustment on the observed frequencies`{O}` blocks the only backdoor path `A ← O → S → T`, so the causal effect is identified by$$P(T=1 \mid do(A=a)) = \sum_{o \in \{0,1\}} P(T=1 \mid A=a, O=o)\, P(O=o)$$Applied directly to the observed cell frequencies, with no model and no prior, this is theprimary estimate of the thesis.

In [ ]:
def backdoor_ate(df, adjust, treatment="A", outcome="T", overlap_only=True, verbose=True):    """    Backdoor adjustment on observed frequencies.    Weights the within-stratum contrast by the empirical distribution of the    adjustment set. With overlap_only=True, strata in which the treatment does not    take both values are dropped instead of being filled in: no data, no estimate.    """    num, den, dropped = 0.0, 0, []    for values in itertools.product((0, 1), repeat=len(adjust)):        g = df        for col, v in zip(adjust, values):            g = g[g[col] == v]        if len(g) == 0:            continue        both = (g[treatment] == 1).any() and (g[treatment] == 0).any()        if not both:            dropped.append((values, len(g)))            if overlap_only:                continue        contrast = (g[g[treatment] == 1][outcome].mean()                    - g[g[treatment] == 0][outcome].mean())        num += len(g) * contrast        den += len(g)    if verbose and dropped:        for values, n in dropped:            cells = ", ".join(f"{c}={v}" for c, v in zip(adjust, values))            print(f"  stratum ({cells}) has no overlap in {treatment} "                  f"— {n} episodes excluded")    return num / den, denate_backdoor_O, n_used = backdoor_ate(data, adjust=["O"])# The same figure, written out as in the thesis.pO = data["O"].mean()do1 = sum((1 - pO if o == 0 else pO) * data[(data.A == 1) & (data.O == o)]["T"].mean()          for o in (0, 1))do0 = sum((1 - pO if o == 0 else pO) * data[(data.A == 0) & (data.O == o)]["T"].mean()          for o in (0, 1))print(f"P(T=1 | do(A=1)) = {do1:.3f}")print(f"P(T=1 | do(A=0)) = {do0:.3f}")print(f"backdoor ATE {{O}} = {ate_backdoor_O:+.3f}   (episodes used: {n_used})")assert abs((do1 - do0) - ate_backdoor_O) < 1e-9RESULTS.append(("chain", "{O}", "observed frequencies", ate_backdoor_O))

The sign flips. Once the confounding influence of the static obstacles is removed, thesignal helps rather than harms.The reason the naive figure is not merely imprecise but actively misleading is a textbookSimpson's paradox. With a clear corridor the task succeeds almost regardless of the actionand the robot rarely acts; with obstacles the context is harder, the success rate is lower,and the robot acts far more often. Pooling the two regimes makes the action lookassociated with failure, when what drives both the higher action rate and the lowersuccess rate is the difficulty of the context.

In [ ]:
def plot_simpson(df, path=None):    """Regenerates Figure 5.1 of the thesis."""    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))    colours = {0: "#1f77b4", 1: "#d62728"}    labels = {0: "$O=0$ (clear)", 1: "$O=1$ (obstacles)"}    # (a) success rate by action, within each level of O and aggregated    for o in (0, 1):        ys, ns = [], []        for a in (0, 1):            g = df[(df.O == o) & (df.A == a)]            ys.append(g["T"].mean())            ns.append(len(g))        ax1.plot([0, 1], ys, "o-", color=colours[o], label=labels[o])        for a in (0, 1):            ax1.annotate(f"{ys[a]:.2f}\n$n$={ns[a]}", (a, ys[a]),                         textcoords="offset points", xytext=(0, 12),                         ha="center", fontsize=8, color=colours[o])    ys = [df[df.A == a]["T"].mean() for a in (0, 1)]    ax1.plot([0, 1], ys, "o--", color="black", label="aggregated")    for a in (0, 1):        ax1.annotate(f"{ys[a]:.2f}", (a, ys[a]), textcoords="offset points",                     xytext=(0, -18), ha="center", fontsize=8)    ax1.annotate(f"naive\nATE $=${ys[1] - ys[0]:+.2f}", (1.02, np.mean(ys)),                 fontsize=8, va="center")    ax1.set_xticks([0, 1])    ax1.set_xticklabels(["$A=0$\n(no action)", "$A=1$\n(signal)"])    ax1.set_ylabel("$P(T=1)$")    ax1.set_ylim(-0.08, 1.18)    ax1.set_title("(a) success rate by action", fontsize=10)    ax1.grid(axis="y", ls=":", alpha=0.5)    # (b) composition of each action group    for a in (0, 1):        g = df[df.A == a]        share1 = (g.O == 1).mean()        ax2.bar(a, 1 - share1, color=colours[0])        ax2.bar(a, share1, bottom=1 - share1, color=colours[1])        ax2.text(a, (1 - share1) / 2, f"{1 - share1:.0%}", ha="center",                 va="center", color="white", fontsize=9)        ax2.text(a, 1 - share1 / 2, f"{share1:.0%}", ha="center",                 va="center", color="white", fontsize=9)        ax2.text(a, 1.03, f"$n$={len(g)}", ha="center", fontsize=8)    ax2.set_xticks([0, 1])    ax2.set_xticklabels(["$A=0$", "$A=1$"])    ax2.set_ylabel("share of episodes")    ax2.set_ylim(0, 1.12)    ax2.set_title("(b) composition of each group", fontsize=10)    handles, lbls = ax1.get_legend_handles_labels()    fig.legend(handles, lbls, loc="lower center", ncol=3, frameon=False,               bbox_to_anchor=(0.5, -0.06))    fig.tight_layout()    if path:        fig.savefig(path, bbox_inches="tight")        print(f"saved {path}")    return fig_ = plot_simpson(data, path="simpson.pdf")plt.show()

### 2.2 The same effect propagated along the chainThe adjustment above uses only the cells of the contingency table. The alternative is tofit the conditional probability tables of the full chain DAG and propagate theintervention along `A → Pe → S → T`, marginalising `O`. The two target the same quantity,the total effect of `A` on `T`, and would agree if the chain were exactly specified; thegap between them measures how far the data depart from that constraint, together with theinfluence of the smoothing prior on the intermediate tables.The estimator below fits every table with an add-$\alpha$ (Laplace) prior, matchingpyAgrum's `useSmoothingPrior()`, so the result is reproducible without pyAgrum.

In [ ]:
def cpt(df, child, parents, alpha=1.0):    """P(child=1 | parents) for every parent configuration, add-alpha smoothed."""    table = {}    for values in itertools.product((0, 1), repeat=len(parents)):        g = df        for col, v in zip(parents, values):            g = g[g[col] == v]        table[values] = ((g[child] == 1).sum() + alpha) / (len(g) + 2 * alpha)    return tabledef chain_ate(df, pi_to_pe=False, alpha=1.0):    """    P(T=1 | do(A=a)) on the chain DAG Pi -> A -> Pe -> S -> T with O -> A, O -> S,    optionally augmented with Pi -> Pe.    Under do(A) the table P(A | Pi, O) is severed and drops out of the computation,    which is why this estimator is unaffected by the empty stratum of Section 3.    """    p_o = [(df.O == 0).mean(), (df.O == 1).mean()]    p_pi = [(df.Pi == 0).mean(), (df.Pi == 1).mean()]    pe_parents = ["A", "Pi"] if pi_to_pe else ["A"]    c_pe = cpt(df, "Pe", pe_parents, alpha)    c_s = cpt(df, "S", ["Pe", "O"], alpha)    c_t = cpt(df, "T", ["S"], alpha)    out = {}    for a in (0, 1):        total = 0.0        for o in (0, 1):            for pi in (0, 1):                w = p_o[o] * (p_pi[pi] if pi_to_pe else 1.0)                key = (a, pi) if pi_to_pe else (a,)                for pe in (0, 1):                    p_pe = c_pe[key] if pe == 1 else 1 - c_pe[key]                    for s in (0, 1):                        p_s = c_s[(pe, o)] if s == 1 else 1 - c_s[(pe, o)]                        total += w * p_pe * p_s * c_t[(s,)]                if not pi_to_pe:                    break        out[a] = total    return out[1] - out[0]ate_chain_O = chain_ate(data)print(f"chain ATE, adjustment {{O}} = {ate_chain_O:+.3f}")RESULTS.append(("chain", "{O}", "propagated, smoothing prior", ate_chain_O))

---## 3. The `Pi → Pe` variant and the overlap violationThesis Section 5.3. If the initial crowd state also influences the downstream chainthrough an edge `Pi → Pe`, the correct adjustment set becomes `{Pi, O}` rather than `{O}`.Before estimating anything under that set, the overlap condition has to be checked: theadjustment is only valid where the treatment takes both values.

In [ ]:
overlap = (data.groupby(["Pi", "O"])["A"]               .agg(episodes="count", A1="sum")               .reset_index())overlap["A0"] = overlap["episodes"] - overlap["A1"]overlap["overlap"] = np.where((overlap.A1 == 0) | (overlap.A0 == 0), "NO", "yes")overlap = overlap[["Pi", "O", "episodes", "A0", "A1", "overlap"]]print(overlap.to_string(index=False))empty = overlap[overlap.overlap == "NO"]n_empty = int(empty["episodes"].sum())print(f"\n{n_empty} of {len(data)} episodes ({n_empty / len(data):.0%}) lie in a stratum "      f"where the contrast cannot be formed.")

The stratum `Pi=0, O=0, A=1` is empty, and it is not a marginal cell: it holds 36 of the100 episodes. This is the policy behaving exactly as designed — with neither a crowd noran obstacle the corridor is never perceived as congested, so the robot never signals — butit means the counterfactual *what would have happened had the robot acted in this easycontext* is not estimable from these data at all.

In [ ]:
ate_backdoor_PiO, n_overlap = backdoor_ate(data, adjust=["Pi", "O"])print(f"backdoor ATE {{Pi,O}} = {ate_backdoor_PiO:+.3f}   "      f"(overlap cells only, {n_overlap} episodes)\n")ate_chain_PiO = chain_ate(data, pi_to_pe=True)print(f"chain ATE {{Pi,O}}    = {ate_chain_PiO:+.3f}")RESULTS.append(("with Pi->Pe", "{Pi,O}", f"observed frequencies, {n_overlap} overlap ep.",                ate_backdoor_PiO))RESULTS.append(("with Pi->Pe", "{Pi,O}", "propagated, smoothing prior", ate_chain_PiO))

Both are positive, and the first is somewhat larger than the estimate adjusting for `O`alone.### 3.1 What happens if the empty stratum is filled in anywayIf the missing cell is not excluded but filled by a smoothing prior in a model where `T`depends directly on `{A, Pi, O}`, the prior invents the counterfactual`P(T=1 | A=1, Pi=0, O=0)` and pulls it towards 0.5 — in a stratum where the observedsuccess rate is close to 1. The result is a fabricated negative effect.The thesis reports −0.122 for this quantity, obtained through pyAgrum; the reconstructionbelow gives −0.123. The difference lies in how the prior is spread over the parentconfigurations and is immaterial to the point being made.

In [ ]:
def smoothed_direct_ate(df, adjust, alpha=1.0):    """T modelled directly on {A} + adjust, every cell smoothed, nothing excluded."""    total = 0.0    for values in itertools.product((0, 1), repeat=len(adjust)):        g = df        for col, v in zip(adjust, values):            g = g[g[col] == v]        w = len(g) / len(df)        p = {}        for a in (0, 1):            cell = g[g.A == a]            p[a] = ((cell["T"] == 1).sum() + alpha) / (len(cell) + 2 * alpha)        total += w * (p[1] - p[0])    return totalate_artefact = smoothed_direct_ate(data, adjust=["Pi", "O"])print(f"{{Pi,O}} with smoothing over the empty stratum = {ate_artefact:+.3f}")print("\nThis is an artefact of the prior, not a signal: it is the only negative adjusted")print("estimate produced anywhere in this analysis, and it disappears once the")print("non-overlapping stratum is excluded, as in the estimate above.")RESULTS.append(("with Pi->Pe", "{Pi,O}", "smoothing over the empty stratum", ate_artefact))

The chain estimator of Section 2.2 is immune to the same problem: under `do(A)` the onlytable touched by the empty stratum is `P(A | Pi, O)`, which is severed by theintervention and drops out of the computation. That is why the chain estimate stayspositive under the very same prior.---## 4. Validation through causal discoveryThesis Section 5.4. The DAG is assumed from domain knowledge, not learned. This sectionasks the independent question of whether the same structure would be recovered from theepisodes alone. It is corroboration, not derivation.

In [ ]:
if HAS_CAUSALLEARN:    cg = pc(data[NODES].values, alpha=0.05, indep_test=chisq,            node_names=NODES, show_progress=False)    g = cg.G.graph    print("PC (alpha=0.05, chi-square), unconstrained:")    found = False    for i, j in itertools.combinations(range(len(NODES)), 2):        if g[i, j] == -1 and g[j, i] == 1:            print(f"  {NODES[i]} -> {NODES[j]}"); found = True        elif g[i, j] == 1 and g[j, i] == -1:            print(f"  {NODES[j]} -> {NODES[i]}"); found = True        elif g[i, j] == -1 and g[j, i] == -1:            print(f"  {NODES[i]} -- {NODES[j]}  (undirected)"); found = True        elif g[i, j] == 1 and g[j, i] == 1:            print(f"  {NODES[i]} <-> {NODES[j]}  (bidirected)"); found = True    if not found:        print("  (no edge recovered)")    print("\nAt 100 episodes and with six binary variables the conditional independence")    print("tests PC relies on are underpowered in the sparser strata, so a partial")    print("recovery is expected. The constrained score-based search below is the")    print("informative one.")else:    print("causal-learn not installed — PC step skipped.")

In [ ]:
ASSUMED_EDGES = [("Pi", "A"), ("O", "A"), ("A", "Pe"), ("Pe", "S"), ("O", "S"), ("S", "T")]VARIANT_EDGE = ("Pi", "Pe")if HAS_PYAGRUM:    data.to_csv("/tmp/bn_data.csv", index=False)    learner = gum.BNLearner("/tmp/bn_data.csv")    # Constraints encode the experimental design, not the relationships under test:    # O and Pi are exogenous, T is a leaf.    for n in NODES:        if n != "O":            learner.addForbiddenArc(n, "O")        if n != "Pi":            learner.addForbiddenArc(n, "Pi")        if n != "T":            learner.addForbiddenArc("T", n)    learner.useGreedyHillClimbing()    bn_learned = learner.learnBN()    learned = {(bn_learned.variable(a[0]).name(), bn_learned.variable(a[1]).name())               for a in bn_learned.arcs()}    print("Hill-climbing (constrained) — learned arcs:")    for a, b in sorted(learned):        print(f"  {a} -> {b}")    print("\nAgainst the assumed structure:")    for e in ASSUMED_EDGES:        print(f"  {e[0]:>2} -> {e[1]:<2}  {'recovered' if e in learned else 'NOT recovered'}")    print(f"  {VARIANT_EDGE[0]:>2} -> {VARIANT_EDGE[1]:<2}  "          f"{'recovered' if VARIANT_EDGE in learned else 'NOT recovered'}  (variant edge)")    extra = learned - set(ASSUMED_EDGES) - {VARIANT_EDGE}    if extra:        print("\nProposed but not assumed:")        for a, b in sorted(extra):            print(f"  {a} -> {b}")else:    print("pyAgrum not installed — hill-climbing step skipped.")    print("Reported in the thesis: every assumed edge is recovered, together with the")    print("variant edge Pi -> Pe; the one divergence is a direct Pe -> T in place of S -> T.")

Hill-climbing recovers the assumed edges and, notably, also the `Pi → Pe` edge of thevariant, which was not obvious a priori. The one substantive divergence concerns theedges into `T`: a direct `Pe → T` is proposed in place of `S → T`. Two targetedconditional independence tests settle it.- `Pe ⊥ T | S` — if the effect of `Pe` on `T` really passes through `S`, independence  should hold and the proposed edge is spurious.- `A ⊥ S | Pe` — if independence fails, there is a path from `A` to `S` bypassing `Pe`,  i.e. a genuinely missing edge.

In [ ]:
def cond_indep_test(df, x, y, z, min_stratum=5):    """Stratified chi-square test of x against y given z."""    stat, dof = 0.0, 0    for value in sorted(df[z].unique()):        sub = df[df[z] == value]        if len(sub) < min_stratum:            continue        table = pd.crosstab(sub[x], sub[y])        if table.shape == (2, 2) and table.values.sum() > 0:            c, _, d, _ = chi2_contingency(table, correction=False)            stat += c            dof += d    p = 1 - chi2.cdf(stat, dof) if dof > 0 else float("nan")    return stat, dof, pfor x, y, z in [("Pe", "T", "S"), ("A", "S", "Pe")]:    s, d, p = cond_indep_test(data, x, y, z)    print(f"{x} vs {y} | {z} :  chi2 = {s:.2f}, dof = {d}, p = {p:.3f}")print("\nPe vs T | S: independence is not rejected, so the effect of Pe on T does pass")print("through S and the Pe -> T edge proposed by hill-climbing is spurious.")print("A vs S | Pe: borderline, and at 100 episodes not conclusive either way.")

In [ ]:
if HAS_PYAGRUM:    cm_learned = csl.CausalModel(bn_learned)    _, p0, _ = csl.causalImpact(cm_learned, on="T", doing="A", values={"A": 0})    _, p1, _ = csl.causalImpact(cm_learned, on="T", doing="A", values={"A": 1})    ate_learned = p1.toarray()[1] - p0.toarray()[1]    print(f"ATE on the learned DAG = {ate_learned:+.3f}")    RESULTS.append(("learned (hill-climbing)", "---",                    "propagated, smoothing prior", ate_learned))else:    print("pyAgrum not installed — estimate on the learned DAG skipped (+0.040 in the thesis).")

---## 5. SummaryThesis Section 5.5 and Table 5.3. Every estimate computed on observed data alone ispositive, in the range +0.03 to +0.11. The naive figure is confounded and sign-inverted;the only other negative value is the artefact of smoothing over the empty stratum.

In [ ]:
summary = pd.DataFrame(RESULTS, columns=["DAG", "adjustment", "method", "ATE"])summary["ATE"] = summary["ATE"].map("{:+.3f}".format)print(summary.to_string(index=False))adjusted_ok = [v for dag, adj, method, v in RESULTS               if adj != "none" and "empty stratum" not in method]print(f"\nAdjusted estimates on observed data: "      f"{min(adjusted_ok):+.3f} to {max(adjusted_ok):+.3f}, all positive.")

---## 6. Cross-check with pyAgrumOptional. The estimates above are computed in plain pandas so that the analysis does notdepend on any causal library. This section repeats the two propagated estimates throughpyAgrum's do-calculus on an explicitly constructed Bayesian network. The two paths shouldagree to within rounding.

In [ ]:
def pyagrum_chain_ate(df, pi_to_pe=False):    bn = gum.BayesNet("HRISim")    for n in NODES:        bn.add(gum.LabelizedVariable(n, n, 2))    arcs = [("Pi", "A"), ("O", "A"), ("A", "Pe"), ("Pe", "S"), ("O", "S"), ("S", "T")]    if pi_to_pe:        arcs.append(("Pi", "Pe"))    for a, b in arcs:        bn.addArc(a, b)    df.to_csv("/tmp/bn_data.csv", index=False)    learner = gum.BNLearner("/tmp/bn_data.csv", bn)    learner.useSmoothingPrior()    bn = learner.learnParameters(bn.dag())    cm = csl.CausalModel(bn)    _, p0, _ = csl.causalImpact(cm, on="T", doing="A", values={"A": 0})    _, p1, _ = csl.causalImpact(cm, on="T", doing="A", values={"A": 1})    return p1.toarray()[1] - p0.toarray()[1]if HAS_PYAGRUM:    for label, pandas_value, kwargs in [        ("chain {O}", ate_chain_O, {}),        ("chain {Pi,O}", ate_chain_PiO, {"pi_to_pe": True}),    ]:        pyagrum_value = pyagrum_chain_ate(data, **kwargs)        delta = abs(pyagrum_value - pandas_value)        flag = "match" if delta < 5e-3 else f"DIFFER by {delta:.4f}"        print(f"{label:<14} pandas {pandas_value:+.3f}   "              f"pyAgrum {pyagrum_value:+.3f}   {flag}")else:    print("pyAgrum not installed — cross-check skipped.")